# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1. Modelos supervisados y métricas (Semana 12)

Profesor: Elias Buitrago Bolivar  
Universidad Central  
Caso: Brazilian E-Commerce Public Dataset by Olist

Objetivo: construir un modelo base para anticipar entregas tardías a partir de datos de compras, productos, clientes, vendedores y pagos.

Las instrucciones de GitHub y entrega se encuentran en el documento técnico. Las secciones de carga y preparación se suministran resueltas; complete las cuatro actividades de modelado y evaluación.

## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [2]:
%pip install scikit-learn


   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.3 MB 5.2 MB/s eta 0:00:02
   ------------- -------------------------- 2.9/8.3 MB 5.6 MB/s eta 0:00:01
   ------------------ --------------------- 3.9/8.3 MB 5.5 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.3 MB 5.5 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.3 MB 5.4 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.3 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 5.4 MB/s eta 0:00:01
   ----------------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Usuario\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import sklearn

print("Versión de Scikit-learn:", sklearn.__version__)

Versión de Scikit-learn: 1.9.1


In [2]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

In [11]:
ORIGEN = ('https://raw.githubusercontent.com/spdrio/'
          'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
          'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')

pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')

print('Pedidos:', pedidos.shape)
print('Items:', items.shape)
print('Productos:', productos.shape)
print('Clientes:', clientes.shape)
print('Vendedores:', vendedores.shape)
print('Pagos:', pagos.shape)

Pedidos: (99441, 8)
Items: (112650, 7)
Productos: (32951, 9)
Clientes: (99441, 5)
Vendedores: (3095, 4)
Pagos: (103886, 5)


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [9]:
print("Columnas de items:")
print(items.columns.tolist())

print("\nColumnas de productos:")
print(productos.columns.tolist())

print("\nColumnas de categorias:")
print(categorias.columns.tolist())

Columnas de items:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_x', 'product_name_lenght_x', 'product_description_lenght_x', 'product_photos_qty_x', 'product_weight_g_x', 'product_length_cm_x', 'product_height_cm_x', 'product_width_cm_x', 'seller_zip_code_prefix_x', 'seller_city_x', 'seller_state_x', 'product_category_name_english_x', 'volumen_cm3', 'categoria', 'product_category_name_y', 'product_name_lenght_y', 'product_description_lenght_y', 'product_photos_qty_y', 'product_weight_g_y', 'product_length_cm_y', 'product_height_cm_y', 'product_width_cm_y', 'seller_zip_code_prefix_y', 'seller_city_y', 'seller_state_y', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'product_category_name_english_y']

Columnas de pro

In [10]:
print("Columnas de items después de las combinaciones:")
print(items.columns.tolist())

Columnas de items después de las combinaciones:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_x', 'product_name_lenght_x', 'product_description_lenght_x', 'product_photos_qty_x', 'product_weight_g_x', 'product_length_cm_x', 'product_height_cm_x', 'product_width_cm_x', 'seller_zip_code_prefix_x', 'seller_city_x', 'seller_state_x', 'product_category_name_english_x', 'volumen_cm3', 'categoria', 'product_category_name_y', 'product_name_lenght_y', 'product_description_lenght_y', 'product_photos_qty_y', 'product_weight_g_y', 'product_length_cm_y', 'product_height_cm_y', 'product_width_cm_y', 'seller_zip_code_prefix_y', 'seller_city_y', 'seller_state_y', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'product_category_name

In [12]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [13]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

Pedidos integrados: 98666


In [14]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

Pedidos disponibles: 96470
Entregas tardías: 6.77 %


### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [15]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

Variables predictoras: 28
Numéricas: 24 | Categóricas: 4


### Muestra individual

Cada estudiante utilizará 35.000 pedidos, seleccionados a partir del ID numérico entregado por el profesor (por ejemplo, ID 01). El mismo ID genera siempre la misma muestra y las mismas particiones.

Se utilizarán 24.500 pedidos para entrenar, 5.250 para validar y 5.250 para la prueba final. Durante las semanas 12 a 15 no se utilizarán los datos de prueba para elegir modelos.

En la siguiente celda cambie `ID_ESTUDIANTE` por el número asignado por el profesor. Por ejemplo, el ID 01 se escribe como `1`.


In [16]:
# Escriba el ID que le asignó el profesor (01 se escribe 1).
ID_ESTUDIANTE = 18
assert isinstance(ID_ESTUDIANTE, int) and 1 <= ID_ESTUDIANTE <= 30, (
    'El ID debe ser un número entero entre 1 y 30.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

Estudiante ID: 18 | Semilla: 1018
Muestra: 35000 | Train: 24500 | Valid: 5250 | Test: 5250
Huella de muestra: 9a47aead0b89


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [27]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)

print('Accuracy referencia:', round(accuracy_score(y_valid, pred_ref), 3))
print('F1 referencia:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

Accuracy referencia: 0.932
F1 referencia: 0.0


### Actividad 1. Entrenamiento

Complete el entrenamiento de la regresión logística usando `preprocesamiento`. Pruebe `class_weight=None` o `'balanced'`, y explique su elección en la sección de interpretación. Conserve las variables `modelo` y `pred`.

En el estimador utilice `class_weight=peso_clases` y `max_iter=1000` como límite de iteraciones. Si aparece una advertencia de convergencia, revise el entrenamiento.


In [21]:
# TODO 1. Entrene el modelo y genere predicciones.

peso_clases = 'balanced'

modelo = make_pipeline(
    preprocesamiento,
    LogisticRegression(
        class_weight=peso_clases,
        max_iter=1000,
        random_state=42
    )
)

modelo.fit(X_train, y_train)

pred = modelo.predict(X_valid)

In [22]:
print("Modelo entrenado:", modelo)
print("\nPrimeras 10 predicciones:")
print(pred[:10])

print("\nCantidad de predicciones:", len(pred))

Modelo entrenado: Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('numericas',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['dias_prometidos',
                                                   'mes_compra',
                                                   'dia_semana_compra',
                                                   'dia_mes_compra',
                                                   'hora_compra',
                                                   'compra_fin_semana',
                                                   'valor_productos',
                                      

## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [24]:
# TODO 2. Calcule las métricas y muestre la matriz de confusión.

# Probabilidad estimada de que cada pedido tenga una entrega tardía.
proba = modelo.predict_proba(X_valid)[:, 1]

# Cálculo de las métricas de evaluación.
medidas = {
    'accuracy': accuracy_score(y_valid, pred),
    'precision': precision_score(y_valid, pred, zero_division=0),
    'recall': recall_score(y_valid, pred, zero_division=0),
    'f1': f1_score(y_valid, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, proba),
    'ap': average_precision_score(y_valid, proba),
}

# Mostrar las métricas.
print("Métricas del modelo:")
for nombre, valor in medidas.items():
    print(f"{nombre}: {valor:.4f}")

# Mostrar la matriz de confusión.
matriz = confusion_matrix(y_valid, pred, labels=[0, 1])

print("\nMatriz de confusión:")
print(matriz)

Métricas del modelo:
accuracy: 0.6592
precision: 0.1178
recall: 0.6225
f1: 0.1981
roc_auc: 0.6965
ap: 0.1456

Matriz de confusión:
[[3240 1655]
 [ 134  221]]


El modelo de regresión logística obtuvo una exactitud del 65,92 % y un recall del 62,25 %, lo que indica que logra identificar una parte importante de las entregas tardías. Sin embargo, la precisión fue del 11,78 %, debido a la cantidad de falsos positivos generados. De los 355 pedidos que realmente tuvieron retrasos, el modelo identificó correctamente 221 y no detectó 134. Además, clasificó erróneamente 1655 pedidos no tardíos como tardíos.

La configuración `class_weight='balanced'` busca compensar el desequilibrio entre las clases y puede favorecer la identificación de los retrasos. No obstante, los resultados muestran que todavía existe un costo importante asociado a las alertas incorrectas. Por esta razón, es necesario evaluar el modelo considerando tanto su capacidad para detectar entregas tardías como los recursos que se necesitarían para atender las alertas generadas.

### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [28]:
# Preparar los datos para analizar los errores
analisis = valid.copy()

analisis['pred'] = pred

analisis['error'] = (
    analisis['is_late'] != analisis['pred']
).astype(int)

print("Variable analisis creada correctamente.")
display(analisis.head())

Variable analisis creada correctamente.


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,...,mes_compra,dia_semana_compra,dia_mes_compra,hora_compra,compra_fin_semana,vendedor_mismo_estado,vendedor_misma_ciudad,proporcion_flete,pred,error
13266,6199bf17a85a73b61d5e43a596b95b5b,a63e6d9ad4758e94c0aa0d3611efc569,delivered,2017-02-02 18:59:30,2017-02-02 19:10:22,2017-02-03 14:36:23,2017-02-08 15:17:35,2017-03-15,SP,torrinha,...,2,3,2,18,0,1,0,0.143744,0,0
29053,d48c5016ab8af7b122d630cd75a63627,31af030fb466a4b0921a9a1ea1d30dfe,delivered,2017-09-15 13:15:51,2017-09-16 02:50:12,2017-09-18 19:27:50,2017-09-25 20:07:49,2017-10-13,MS,nova andradina,...,9,4,15,13,0,0,0,0.137336,0,0
8110,3b55f8bdca319d87fac505893a4a3440,a6a6ab9ca44dfb6bcb8faa2f764cfb6c,delivered,2017-06-15 00:36:09,2017-06-17 03:47:46,2017-06-19 17:29:37,2017-06-30 14:05:55,2017-07-10,RJ,petropolis,...,6,3,15,0,0,0,0,0.407118,1,1
13172,60f6e33dc906f7b5b4f5096f979ab9f6,194210b93d80c9ed81d077596cbd8ffb,delivered,2018-05-09 08:06:14,2018-05-09 08:30:57,2018-05-09 11:00:00,2018-05-15 18:39:28,2018-05-28,RJ,sao francisco de itabapoana,...,5,2,9,8,0,0,0,0.415305,1,1
1083,079aa97bd14929502e11d87d14790792,6e544db78613e468780690fcdd22a7bc,delivered,2018-08-14 17:11:29,2018-08-14 17:30:32,2018-08-15 12:22:00,2018-08-20 17:09:01,2018-09-03,RJ,macae,...,8,1,14,17,0,0,0,0.116500,1,1


In [29]:
# TODO 3. Agrupe por mes_compra y customer_state.
errores_segmento = (
    analisis
    .groupby(['mes_compra', 'customer_state'])
    .agg(
        pedidos=('error', 'size'),
        errores=('error', 'sum'),
        tasa_error=('error', 'mean'),
        retrasos_reales=('is_late', 'sum'),
        retrasos_predichos=('pred', 'sum')
    )
    .reset_index()
)

errores_segmento['tasa_error'] *= 100

errores_segmento = errores_segmento.sort_values(
    by='tasa_error',
    ascending=False
)

display(errores_segmento.head(15))

,mes_compra,customer_state,pedidos,errores,tasa_error,retrasos_reales,retrasos_predichos
254,12,AL,1,1,100.0,0,1
248,11,RN,1,1,100.0,0,1
31,2,MA,3,3,100.0,0,3
24,2,AL,2,2,100.0,0,2
21,1,SE,2,2,100.0,0,2
232,11,AL,3,3,100.0,0,3
230,10,SE,1,1,100.0,0,1
221,10,PA,1,1,100.0,1,0
210,9,TO,1,1,100.0,0,1
208,9,SE,2,2,100.0,0,2


EL objetivo es determinar si los errores se concentran en algún segmento específico y si eso puede orientar mejoras del modelo.

Por ahora, sabemos que existen errores de ambos tipos.
 
Falso positivo: el modelo predice un retraso que no ocurrió.
Falso negativo: el modelo no identifica un retraso que sí ocurrió.

Sin embargo, necesitamos analizar grupos más grandes antes de atribuirlos a un mes o estado particular.

In [30]:
# Filtrar segmentos con al menos 20 pedidos
segmentos_relevantes = errores_segmento[
    errores_segmento['pedidos'] >= 20
].copy()

# Ordenar por tasa de error descendente
segmentos_relevantes = segmentos_relevantes.sort_values(
    by='tasa_error',
    ascending=False
)

display(segmentos_relevantes.head(15))

,mes_compra,customer_state,pedidos,errores,tasa_error,retrasos_reales,retrasos_predichos
159,7,RJ,74,57,77.027027,3,60
161,7,RS,27,20,74.074074,1,19
180,8,RJ,71,51,71.830986,4,55
203,9,RJ,39,27,69.230769,2,29
16,1,RJ,51,35,68.627451,6,35
225,10,RJ,40,27,67.500000,1,28
88,4,RJ,69,46,66.666667,1,45
64,3,RJ,62,40,64.516129,12,48
113,5,RS,30,19,63.333333,0,19
270,12,RJ,44,26,59.090909,7,21


Esto sugiere que está generando numerosos falsos positivos: pedidos que predice como tardíos, aunque en realidad no lo fueron.

En la tabla también aparecen segmentos donde los retrasos reales y los predichos son más cercanos. Por eso, conviene examinar el comportamiento de cada segmento y no solamente ordenar por la tasa de error.

## comparar los resultados agrupando únicamente por estado

In [31]:
# Analizar errores por estado del cliente
errores_estado = (
    analisis
    .groupby('customer_state')
    .agg(
        pedidos=('error', 'size'),
        errores=('error', 'sum'),
        tasa_error=('error', 'mean'),
        retrasos_reales=('is_late', 'sum'),
        retrasos_predichos=('pred', 'sum')
    )
    .reset_index()
)

errores_estado['tasa_error'] *= 100

errores_estado = errores_estado.sort_values(
    by='tasa_error',
    ascending=False
)

display(errores_estado)

,customer_state,pedidos,errores,tasa_error,retrasos_reales,retrasos_predichos
13,PA,51,37,72.549020,3,38
1,AL,29,21,72.413793,6,27
5,CE,81,57,70.370370,10,67
15,PE,80,56,70.000000,8,62
24,SE,28,19,67.857143,3,22
14,PB,31,21,67.741935,3,22
9,MA,39,26,66.666667,9,35
4,BA,167,110,65.868263,22,132
18,RJ,699,442,63.233190,73,485
26,TO,13,8,61.538462,0,8


1. El modelo tiene dificultades en varios estados.

Los estados PA (Pará), AL (Alagoas), CE (Ceará) y PE (Pernambuco) presentan tasas de error cercanas o superiores al 70 %. Sin embargo, hay que tener en cuenta el número de pedidos: por ejemplo, AL solo tiene 29, mientras que CE tiene 81. Por eso, los resultados de CE y PE ofrecen una referencia más estable que los de los estados con muy pocos pedidos.

2. Río de Janeiro (RJ) sigue siendo un segmento importante.

RJ tiene 699 pedidos y 442 errores, con una tasa de error del 63,23 %. Como también aparecía repetidamente entre los segmentos mensuales con más errores, podemos afirmar que el problema no se limita a un solo mes.

Además:
- Retrasos reales: 73.
- Retrasos predichos: 485.
- El modelo predice muchos más retrasos de los que realmente ocurren en este estado.

3. El modelo también presenta diferencias importantes entre estados.

En São Paulo (SP), por ejemplo, la tasa de error es del 20,66 %, mientras que en RJ es del 63,23 %. Esto indica que el desempeño varía considerablemente según el estado del cliente.

No obstante, una tasa de error elevada no demuestra por sí sola que el estado sea la causa. También podrían influir la distribución de los datos, la cantidad de pedidos retrasados y otras características de los pedidos.

## el modelo detecta una parte considerable de los retrasos, pero genera muchas falsas alarmas.

# Comprobar los errores por tipo

In [34]:
errores_estado_detalle = (
    analisis
    .groupby('customer_state')
    .apply(
        lambda grupo: pd.Series({
            'pedidos': len(grupo),
            'falsos_positivos': (
                (grupo['is_late'] == 0) & (grupo['pred'] == 1)
            ).sum(),
            'falsos_negativos': (
                (grupo['is_late'] == 1) & (grupo['pred'] == 0)
            ).sum(),
            'retrasos_reales': grupo['is_late'].sum(),
            'retrasos_predichos': grupo['pred'].sum()
        }),
        include_groups=False
    )
    .reset_index()
)

display(
    errores_estado_detalle.sort_values(
        by='falsos_positivos',
        ascending=False
    ).head(10)
)

,customer_state,pedidos,falsos_positivos,falsos_negativos,retrasos_reales,retrasos_predichos
18,RJ,699,427,15,73,485
25,SP,2212,398,59,102,441
22,RS,273,112,7,15,120
4,BA,167,110,0,22,132
23,SC,190,89,6,22,105
10,MG,613,77,19,25,83
5,CE,81,57,0,10,67
15,PE,80,55,1,8,62
8,GO,122,44,3,9,50
7,ES,94,43,2,12,53


Se puede concluir que el modelo se inclina demasiado por predecir retrasos, ademas que el desempeño varía según el estado sin embargo, todavía no podemos afirmar que el estado del cliente sea la causa de los errores.

Si reducimos las falsas alarmas, podríamos mejorar la precisión del modelo, pero también correríamos el riesgo de dejar pasar más retrasos reales. Asi que debemos encontrar un equilibrio entre Precision y Recall

## Revisar el umbral de clasificación

Al aumentar el umbral, normalmente disminuye la cantidad de pedidos clasificados como retrasados. Esto puede aumentar la precisión, pero reducir el recall.

In [35]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = modelo.predict_proba(X_valid)[:, 1]

for umbral in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_umbral = (proba >= umbral).astype(int)

    precision = precision_score(
        y_valid, pred_umbral, zero_division=0
    )
    recall = recall_score(
        y_valid, pred_umbral, zero_division=0
    )
    f1 = f1_score(
        y_valid, pred_umbral, zero_division=0
    )

    print(
        f"Umbral: {umbral:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Umbral: 0.3 | Precision: 0.0826 | Recall: 0.9155 | F1: 0.1515
Umbral: 0.4 | Precision: 0.0949 | Recall: 0.7915 | F1: 0.1695
Umbral: 0.5 | Precision: 0.1178 | Recall: 0.6225 | F1: 0.1981
Umbral: 0.6 | Precision: 0.1437 | Recall: 0.4225 | F1: 0.2144
Umbral: 0.7 | Precision: 0.1832 | Recall: 0.2451 | F1: 0.2096


## ¿Cuál es el mejor umbral?

Ya que cada metrica analizada significa:
- Precisión: de los pedidos que el modelo predice como retrasados, ¿cuántos realmente lo están?
- Recall: de todos los pedidos que realmente se retrasaron, ¿cuántos consigue detectar el modelo?
- F1-score: combina precisión y recall para evaluar el equilibrio entre ambas.

Si estamos buscando un equilibrio entre ambas metricas el mas acertado es el umbral 0.6, es el mejor según el F1-score entre los valores probados, pero no podemos afirmar que sea el mejor para el proyecto sin definir primero qué error es más costoso. Si la prioridad es advertir a los clientes antes de que sus pedidos se retrasen, perder tantos retrasos reales podría ser contraproducente.


In [36]:
from sklearn.metrics import confusion_matrix

for umbral in [0.5, 0.6]:
    pred_umbral = (
        modelo.predict_proba(X_valid)[:, 1] >= umbral
    ).astype(int)

    matriz = confusion_matrix(
        y_valid,
        pred_umbral,
        labels=[0, 1]
    )

    tn, fp, fn, tp = matriz.ravel()

    print(f"\nUmbral: {umbral}")
    print(f"Verdaderos negativos: {tn}")
    print(f"Falsos positivos:     {fp}")
    print(f"Falsos negativos:     {fn}")
    print(f"Verdaderos positivos: {tp}")


Umbral: 0.5
Verdaderos negativos: 3240
Falsos positivos:     1655
Falsos negativos:     134
Verdaderos positivos: 221

Umbral: 0.6
Verdaderos negativos: 4001
Falsos positivos:     894
Falsos negativos:     205
Verdaderos positivos: 150


- Falsos positivos: pedidos que el modelo predice como retrasados, pero que no lo están.
- Falsos negativos: pedidos que realmente se retrasan, pero que el modelo no identifica.
- Verdaderos positivos: pedidos retrasados que el modelo detecta correctamente.
- Verdaderos negativos: pedidos sin retraso que el modelo identifica correctamente.

## ¿Que umbral conviene utilizar?

Si el objetivo es anticipar la mayor cantidad posible de retrasos, el umbral 0,5 resulta más adecuado entre estas dos opciones, porque detecta 221 retrasos frente a 150.

Si queremos reducir las falsas alarmas, el umbral 0,6 es más conveniente: disminuye los falsos positivos de 1655 a 894, una reducción aproximada del 46 %.

Sin embargo, no debemos interpretar esta reducción de forma aislada. Con el umbral 0,6, los falsos negativos aumentan de 134 a 205, aproximadamente un 53 %.

Por tanto, se conservar el umbral 0,5 como referencia aunque se aclara el umbral 0,6 como una alternativa que reduce falsas alarmas, pero sacrifica la detección de retrasos. Ningun umbral es definitivo sin antes establecer qué tipo de error es más perjudicial para el proyecto.

### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

Utilizamos los valores de umbral 0.5 de falsos positivos como falsos negativos

In [37]:
# TODO 4. Calcule el costo total con la matriz de confusión.
costo_fp_fn = (1655 * 8) + (134 * 35)

print(f"Costo total de los errores: R$ {costo_fp_fn}")

Costo total de los errores: R$ 17930


### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [38]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


Autoverificación superada.


## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [39]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

MAE regresión: 5.04 días


Esto significa que, en promedio, las predicciones de duración de entrega se desvían aproximadamente 5 días respecto a los tiempos reales.

Por ejemplo, si un pedido tarda realmente 15 días en entregarse, el modelo podría predecir 10 o 20 días, aunque el MAE no significa que todos los pedidos tengan exactamente ese margen de error.

# 6. Interpretación de negocio

## 1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.

Respuesta:

La regresión logística, utilizando `class_weight='balanced'`, obtuvo un Accuracy de 65,92 %, un Recall de 62,25 % y un F1-score de 19,81 %. 

Se eligió `class_weight='balanced'` porque existe un desequilibrio entre los pedidos retrasados y los que no presentan retraso, con el propósito de mejorar la detección de retrasos. Sin embargo, el modelo presentó una precisión baja del 11,78 % y numerosos falsos positivos, por lo que muchas de las alertas generadas no corresponden a retrasos reales.

En conclusión, el modelo logra identificar una parte importante de los pedidos retrasados, pero todavía debe mejorar el equilibrio entre la detección de retrasos y la reducción de falsas alarmas, Para elegir la mejor configuración. 

## 2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?

Respuesta:

Los segmentos con mayor tasa de error corresponden principalmente a pedidos de ciertos estados, como RJ (Río de Janeiro), RS (Rio Grande do Sul), CE (Ceará) y PE (Pernambuco), especialmente en algunos meses donde el error superó el 60 %. Al analizar los tipos de error, se observa que predominan los falsos positivos: pedidos que el modelo predice como retrasados, aunque realmente no lo están. 

Esto se relaciona directamente con los costos empresariales establecidos en el ejercicio: cada falso positivo cuesta R$ 8, mientras que cada falso negativo cuesta R$ 35. Por lo tanto, aunque los falsos positivos son más numerosos, los falsos negativos tienen un costo individual mayor, ya que representan retrasos reales que el modelo no logra anticipar. En consecuencia, es necesario encontrar un equilibrio entre reducir las falsas alarmas y detectar la mayor cantidad posible de retrasos, buscando minimizar el costo total de los errores.

## 3. Proponga una mejora concreta para la semana 13. Considere la información del producto, del vendedor, del pago o el tratamiento de variables categóricas. Explique cómo verificará la mejora usando la misma partición de validación.

Respuesta:

Revisar variables relacionadas con el producto y el vendedor, como la categoría del producto, la cantidad de artículos por pedido y el tiempo de preparación del vendedor, siempre que estos datos estén disponibles antes de la predicción. También revisaría el tratamiento de las variables categóricas para asegurar que se codifiquen correctamente y que las categorías poco frecuentes no afecten negativamente el desempeño.

Para verificar la mejora, entrenaría nuevamente la regresión logística manteniendo la misma partición de entrenamiento y validación, y compararía sus resultados con el modelo actual utilizando Accuracy, Recall, F1-score y la matriz de confusión. Además, calcularía el costo empresarial con R$ 8 por cada falso positivo y R$ 35 por cada falso negativo. La mejora se consideraría útil si reduce el costo total de los errores sin disminuir excesivamente la detección de retrasos. De esta manera, la comparación permitiría determinar si los nuevos datos y ajustes realmente mejoran el modelo.

## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [41]:
import sys
import sklearn

reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

ruta = Path(f'reports/S12_resultados_ID_{identificador}.json')
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reporte:', ruta)

# En Colab, descarga el JSON para conservarlo fuera de la sesión.
try:
    from google.colab import files
    files.download(str(ruta))
except Exception:
    print('Si no se descargó automáticamente, guarde manualmente:', ruta)


Reporte: reports\S12_resultados_ID_18.json
Si no se descargó automáticamente, guarde manualmente: reports\S12_resultados_ID_18.json
